# Two-Layer Acoustic and Elastic Propagation

This notebook demonstrates running forward acoustic and elastic IWAVE propagations using `run_iwave` with a simple two-layer model. The IWAVE binaries (`acd` and `asg`) must be built and available on the system path.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
repo_root = Path(os.environ['REPO_ROOT'])
trip_root = repo_root / 'trip'
import sys
sys.path.insert(0, str(repo_root))
from trip.iwave.python.iwave import run_iwave


In [ ]:
trip_root = Path().resolve().parents[3]
ACD_BIN = trip_root / "iwave" / "acd" / "main" / "acd.x"
ASG_BIN = trip_root / "iwave" / "asg" / "main" / "asg.x"


In [ ]:
# Model parameters
nx, nz = 40, 40
dx = dz = 10.0
nt = 40
dt = 0.004

z = np.arange(nz)
vp = np.where(z < nz//2, 1500.0, 2000.0)
vp = np.repeat(vp[:, None], nx, axis=1)

rho = np.where(z < nz//2, 1000.0, 1200.0)
rho = np.repeat(rho[:, None], nx, axis=1)

# Acoustic model uses squared velocity in km/s
csq = (vp/1000.0)**2

# Source: Ricker wavelet at surface center
f0 = 10.0
t = np.arange(nt)*dt
wavelet = (1 - 2*(np.pi*f0*t)**2)*np.exp(-(np.pi*f0*t)**2)
source = np.zeros((nt, nx), dtype=np.float32)
source[:, nx//2] = wavelet


In [ ]:
# Forward acoustic propagation
acoustic_inputs = {
    'csq': csq.astype('float32'),
    'source': source,
}
acoustic_outputs = run_iwave(
    str(ACD_BIN),
    acoustic_inputs,
    output_specs={'data': source.shape},
    extra_args=[f'nt={nt}', f'dt={dt}', f'dx={dx}', f'dz={dz}', 'deriv=0', f'cmin={vp.min()/1000}', f'cmax={vp.max()/1000}'],
)
acoustic_data = acoustic_outputs['data']


In [ ]:
plt.figure(figsize=(6,4))
plt.imshow(acoustic_data.T, aspect='auto', cmap='seismic', origin='lower')
plt.title('Acoustic data')
plt.xlabel('Trace')
plt.ylabel('Time sample')
plt.show()


In [ ]:
# Elastic parameters
vs = 0.6 * vp
lam = rho * (vp**2 - 2*vs**2)
mu = rho * vs**2

elastic_inputs = {
    'lambda': lam.astype('float32'),
    'mu': mu.astype('float32'),
    'rho': rho.astype('float32'),
    'source': source,
}
elastic_outputs = run_iwave(
    str(ASG_BIN),
    elastic_inputs,
    output_specs={'data': source.shape},
    extra_args=[f'nt={nt}', f'dt={dt}', f'dx={dx}', f'dz={dz}', 'deriv=0', f'cmin={vs.min()/1000}', f'cmax={vp.max()/1000}', f'dmin={1/rho.max()/1000}', f'dmax={1/rho.min()/1000}'],
)
elastic_data = elastic_outputs['data']


In [ ]:
plt.figure(figsize=(6,4))
plt.imshow(elastic_data.T, aspect='auto', cmap='seismic', origin='lower')
plt.title('Elastic data')
plt.xlabel('Trace')
plt.ylabel('Time sample')
plt.show()
